In [16]:
import numpy as np
import gymnasium as gym
import os
import torch
import torch.nn as nn
import sys
SCRIPT_DIR = os.path.dirname(os.path.abspath("..src"))
sys.path.append(os.path.dirname(SCRIPT_DIR))
from src.wrappers.env_wrappers import ChannelsObs, StochasticAction
# from src.replay_buffer import MonReplayBuffer

In [14]:
from stable_baselines3 import DQN
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

In [15]:
class CustomCNN(BaseFeaturesExtractor):
    """
    :param observation_space: (gym.Space)
    :param features_dim: (int) Number of features extracted.
        This corresponds to the number of unit for the last layer.
    """

    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 64):
        super().__init__(observation_space, features_dim)
        # We assume CxHxW images (channels first)
        # Re-ordering will be done by pre-preprocessing or wrapper
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 8, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(8, 4, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with torch.no_grad():
            n_flatten = self.cnn(
                torch.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]
        # print("n_flatten", n_flatten) # 100
        # self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())
        self.linear = nn.Linear(n_flatten, features_dim)

    def forward(self, observations) -> torch.Tensor:
        if isinstance(observations, np.ndarray):
            obervation = torch.from_numpy(observations.reshape(1, 3, 9, 9))
        return self.linear(self.cnn(observations))

In [16]:
policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=64),
)

In [ ]:
gamma = 0.99
env = gym.make("gym_monitor/TreasureHunt-Penalty-v1", render_modes="human", seed=0)
env = ChannelsObs(env, grid_size=(9,9), n_objects=3)
# env = StochasticAction(env, random_prob=0.1)

model = DQN(
    "CnnPolicy",env, policy_kwargs=policy_kwargs, verbose=1, device="mps:0", learning_starts=int(1e5), learning_rate=1e-3, gamma=gamma, exploration_fraction=0.5,
    # replay_buffer_class=ReplayBufferCustom,
    tensorboard_log="models/dqn/env_0.0/mdp_seed_0/start_100k/test")
model.learn(total_timesteps=int(5e5), log_interval=50)
# model.save("models/dqn/env_0.0/mdp_seed_0/start_100k")
# model.load("models/dqn/env_0.0/mdp_seed_0/start_100k.zip")

In [ ]:
for ep in range(10):
    new_env = model.get_env()
    s = new_env.reset()
    R = 0
    for i in range(5000):
        s = torch.tensor(s.reshape(1, 3, 9, 9), dtype=torch.float32)#.to("mps:0")
        action, _ = model.predict(s, deterministic=True)
        # print(action)
        s, r, done, info = new_env.step(action)
        # new_env.render("human")
        R += r * (gamma ** i)
        if done:
            break
    print("reward", R, i + 1)

In [ ]:
model.load_replay_buffer("models/dqn/env_0.0/mdp_seed_0/start_100k/DQN_2/")

In [8]:
import gymnasium as gym
import numpy as np
import os
import sys
SCRIPT_DIR = os.path.dirname(os.path.abspath("..src"))
sys.path.append(os.path.dirname(SCRIPT_DIR))
from src.wrappers.env_wrappers import ChannelsObs, TabularObservationsWrapper
from src.critic import MonQNet, MonQTableOneAction
env = gym.make("gym_monitor/TreasureHunt-Penalty-v1", render_modes="human", seed=0)
# env = ChannelsObs(env, grid_size=(9,9), n_objects=3)
env = TabularObservationsWrapper(env, grid_size=(9,9))

In [4]:
critic = MonQNet(env_name="gym_monitor/TreasureHunt-Penalty-v1", observation_space=env.observation_space, action_space=env.action_space, dir_name="")

In [9]:
critic = MonQTableOneAction("gym_monitor/TreasureHunt-Penalty-v1", env.observation_space, env.action_space, "")

TypeError: 'Discrete' object is not subscriptable

In [5]:
critic.reset()

AttributeError: 'MonQNet' object has no attribute '_q_network'

In [14]:
s, _ = env.reset()
s

array([[[1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0

In [64]:
env.step(3)

(array([[[0., 0., 0., 0., 0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.]],
 
        [[0., 0., 0., 0., 0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0.]],
 
        [[0., 0., 0., 0., 1., 0., 0., 0., 0.],
         [0., 0., 0., 0., 1., 0., 0., 0., 0.],
         [0., 0., 0., 0., 1., 0., 0., 0., 0.],
       

In [14]:
env.get_grid()

array([[0, 0, 0, 0, 3, 0, 0, 0, 2],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 3, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [6]:
env.get_agent_pos()

(1, 0)

In [6]:
list(env.get_agent_pos())

[1, 0]

In [7]:
np.where(s == 1)[0]

array([0])